In [1]:
"""
==============================================================
TRAINING RANDOM FOREST - Deteksi Stunting Pada Balita
==============================================================
Input    : dataset_clean.csv (hasil data_preparation.py)
Output   : model_stunting.pkl
Fitur    : JK, Usia_Bulan, Berat, Tinggi
Target   : Status_Stunting (Normal / Pendek / Sangat Pendek)
==============================================================
"""

import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score)

# ──────────────────────────────────────────────
# FASE 1: LOAD DATASET BERSIH
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATASET BERSIH")
print("=" * 55)

df = pd.read_csv('../data/processed/dataset_clean.csv')
print(f"✅ Dataset dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"   Kolom: {df.columns.tolist()}\n")


# ──────────────────────────────────────────────
# FASE 2: PISAH FITUR DAN TARGET
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 2: PISAH FITUR DAN TARGET")
print("=" * 55)

feature_cols = ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI', 'Rasio_BB_TB']
X = df[feature_cols]
y_raw = df['Status_Stunting']

# Encode label target
le = LabelEncoder()
y = le.fit_transform(y_raw)

print(f"✅ Fitur (X) : {feature_cols}")
print(f"✅ Target (y): {y_raw.unique().tolist()}")
print(f"   Encoded   : Normal=0 | Pendek=1 | Sangat Pendek=2")
print(f"   Shape X   : {X.shape}\n")


# ──────────────────────────────────────────────
# FASE 3: SPLIT TRAIN & TEST (80:20)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 3: SPLIT TRAIN & TEST (80:20 Stratified)")
print("=" * 55)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Data training : {X_train.shape[0]} baris")
print(f"✅ Data testing  : {X_test.shape[0]} baris\n")


# ──────────────────────────────────────────────
# FASE 4: TRAINING MODEL RANDOM FOREST
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 4: TRAINING MODEL RANDOM FOREST")
print("=" * 55)

rf_model = RandomForestClassifier(
    n_estimators=200,       # jumlah pohon
    max_depth=20,         # kedalaman pohon (None = bebas)
    min_samples_split=5,    # min sampel untuk split node
    min_samples_leaf=2,     # min sampel di leaf node
    max_features='sqrt',  
    class_weight='balanced',# tangani class imbalance
    random_state=42,
    n_jobs=-1               # pakai semua core CPU
)

rf_model.fit(X_train, y_train)
print(f"✅ Model berhasil ditraining!")
print(f"   Jumlah pohon    : {rf_model.n_estimators}")
print(f"   Class weight    : balanced\n")


# ──────────────────────────────────────────────
# FASE 5: EVALUASI MODEL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 5: EVALUASI MODEL")
print("=" * 55)

y_pred = rf_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"✅ Akurasi pada data test: {acc * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,
    index=[f'Aktual: {c}' for c in le.classes_],
    columns=[f'Prediksi: {c}' for c in le.classes_]
)
print(cm_df.to_string())
print()

# ──────────────────────────────────────────────
# FASE 6: CROSS VALIDATION (5-Fold)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 6: CROSS VALIDATION (5-Fold)")
print("=" * 55)

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='accuracy')

print(f"✅ CV Scores tiap fold : {[f'{s*100:.2f}%' for s in cv_scores]}")
print(f"   Rata-rata CV Score  : {cv_scores.mean()*100:.2f}%")
print(f"   Std Deviasi         : {cv_scores.std()*100:.2f}%\n")

# ──────────────────────────────────────────────
# FASE 7: FEATURE IMPORTANCE
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7: FEATURE IMPORTANCE")
print("=" * 55)

importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False)

print("Kontribusi tiap fitur terhadap model:")
for feat, imp in feat_imp.items():
    bar = 'X' * int(imp * 50)
    print(f"   {feat:<12}: {imp:.4f} ({imp*100:.1f}%) {bar}")
print()


# ──────────────────────────────────────────────
# FASE 8: SIMPAN MODEL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 8: SIMPAN MODEL")
print("=" * 55)

# Simpan model RF
with open("../models/model_stunting.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("../models/label_encoder_stunting.pkl", "wb") as f:
    pickle.dump(le, f)

print("✅ File tersimpan:")
print("   📦 model_stunting.pkl          → Model Random Forest")
print("   📦 label_encoder_stunting.pkl  → Label encoder (decode prediksi)\n")


# ──────────────────────────────────────────────
# FASE 9: UJI PREDIKSI MANUAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 9: UJI PREDIKSI MANUAL")
print("=" * 55)

# Contoh: balita perempuan, 24 bulan, BB 9kg, TB 80cm
# ZS BB/U, ZS TB/U, ZS BB/TB diisi contoh

berat = 9.0
tinggi = 80.0

contoh_data = pd.DataFrame([{
    'JK': 0,           # P=0
    'Usia_Bulan': 24,
    'Berat': 9.0,
    'Tinggi': 80.0,
    'BMI': berat / ((tinggi / 100) ** 2),       # hitung otomatis
    'Rasio_BB_TB': berat / tinggi,  
}])

hasil_encoded = rf_model.predict(contoh_data)[0]
hasil_label   = le.inverse_transform([hasil_encoded])[0]
probabilitas  = rf_model.predict_proba(contoh_data)[0]

print("Input contoh balita:")
print(f"   JK=P, Usia=24 bln, BB={berat}kg, TB={tinggi}cm")
print(f"   BMI={contoh_data['BMI'].values[0]:.2f} | Rasio BB/TB={contoh_data['Rasio_BB_TB'].values[0]:.4f}")
print(f"\nHasil Prediksi : {hasil_label}")
print(f"Probabilitas   :")
for kelas, prob in zip(le.classes_, probabilitas):
    print(f"   {kelas:<15}: {prob*100:.1f}%")

FASE 1: LOAD DATASET BERSIH
✅ Dataset dimuat: 2439 baris, 7 kolom
   Kolom: ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI', 'Rasio_BB_TB', 'Status_Stunting']

FASE 2: PISAH FITUR DAN TARGET
✅ Fitur (X) : ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI', 'Rasio_BB_TB']
✅ Target (y): ['Sangat Pendek', 'Normal', 'Pendek']
   Encoded   : Normal=0 | Pendek=1 | Sangat Pendek=2
   Shape X   : (2439, 6)

FASE 3: SPLIT TRAIN & TEST (80:20 Stratified)
✅ Data training : 1951 baris
✅ Data testing  : 488 baris

FASE 4: TRAINING MODEL RANDOM FOREST
✅ Model berhasil ditraining!
   Jumlah pohon    : 200
   Class weight    : balanced

FASE 5: EVALUASI MODEL
✅ Akurasi pada data test: 86.27%

Classification Report:
               precision    recall  f1-score   support

       Normal       0.95      0.93      0.94       308
       Pendek       0.69      0.77      0.73        91
Sangat Pendek       0.74      0.72      0.73        89

     accuracy                           0.86       488
    macro avg       0